# 🚀 SmartVision AI – Automated Dataset Collection & YOLOv8 GPU Training Pipeline

This Google Colab notebook automates the entire end-to-end vision pipeline:
1. **GPU Acceleration**: Leverage free T4 / A100 GPU compute.
2. **Dataset Discovery & Automated Download**: Discover paginated Kaggle & Hugging Face datasets.
3. **Multi-Format Preparation**: Automatically convert VOC XML, COCO JSON, and YOLO annotations to standard 80 canonical classes.
4. **YOLOv8 Training**: Fine-tune YOLOv8 models (`yolov8n` or `yolov8m`).
5. **Rigorous Benchmark Evaluation**: Evaluate on held-out test split (Precision, Recall, mAP50, mAP50-95).
6. **Model Export**: Export production weights to ONNX and TFLite.

### Step 1: Verify Hardware Accelerator (GPU)

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

### Step 2: Clone or Sync SmartVision Project

In [ ]:
import os
# Clone the repository if not already in workspace
if not os.path.exists('SmartVision'):
    !git clone https://github.com/soumyajali/SmartVision.git
    %cd SmartVision
else:
    %cd SmartVision

# Install dependencies
!pip install -q ultralytics huggingface_hub kaggle pyyaml onnx onnxruntime

### Step 3: Kaggle API Authentication Setup (Optional for Kaggle Datasets)
Set your Kaggle credentials securely using Colab Secrets or environment variables.
*(If not configured, the pipeline automatically proceeds with Hugging Face and benchmark datasets)*

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✓ Kaggle credentials loaded from Colab Secrets!")
except Exception:
    print("ℹ Colab Secrets not set. You can set them manually below if downloading from Kaggle:")
    # os.environ['KAGGLE_API_TOKEN'] = 'your_token_here'

### Step 4: Run Automated Dataset Discovery & Batch Download

In [ ]:
# Discover datasets from Kaggle & Hugging Face
!python scripts/dataset_pipeline.py --discover

# Download top matching datasets within storage limits
!python scripts/dataset_pipeline.py --download

### Step 5: Convert Annotations, Validate, & Create Leak-Free Splits

In [ ]:
!python scripts/dataset_pipeline.py --prepare

### Step 6: High-Performance GPU Training (YOLOv8)

In [ ]:
# Train for 20 epochs with GPU acceleration
!python scripts/dataset_pipeline.py --train --epochs 20

### Step 7: Benchmark Evaluation & Baseline Comparison

In [ ]:
!python scripts/dataset_pipeline.py --evaluate

### Step 8: Export Production Model (ONNX & TFLite) & Download

In [ ]:
# Export trained weights to ONNX and TFLite
!python scripts/dataset_pipeline.py --export

# Download the best model weights to your local machine
from google.colab import files
if os.path.exists('weights/smartvision_best.pt'):
    files.download('weights/smartvision_best.pt')
if os.path.exists('weights/smartvision_best.onnx'):
    files.download('weights/smartvision_best.onnx')